← [Overview](00_overview.ipynb)

# Representation

Clustering decides **which** periods group together. Representation decides **what each
group's single profile looks like**. In tsam these are separate, recombinable steps: every
clustering method sets a default representation, and you can override it freely — Ward with a
maxoid, k-means with a medoid, any pairing you like.

A representation rule acts on **one cluster at a time**, independently of every other cluster.
So one cluster is the whole story, and that is what this notebook follows: four days go in,
six rules act on them, one profile comes out of each.

| | |
|---|---|
| **In** | the periods of one cluster — here four days of the [tiny six-day set](01_preprocessing.ipynb) |
| **Inside** | one of six rules for condensing them into a single profile |
| **Out** | one profile per cluster: `n_timesteps × n_attributes` numbers |

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import tsam
from tsam import ClusterConfig
from tsam.config import MinMaxMean

pio.renderers.default = "notebook_connected"

ATTRS = ["solar", "load"]
UNITS = {"solar": "W/m²", "load": "MW"}
N_TIMESTEPS = 4

# The raw six-day series and the preprocessed period matrix D, both written by
# 01_preprocessing. D is normalized and unstacked: one row per day, columns
# (attribute, timestep). The rules below operate on D — the same matrix the
# clustering notebooks cluster.
tiny = pd.read_csv("../../data/tiny.csv", index_col=0, parse_dates=True)
D = pd.read_csv("../../data/tiny_periods.csv", header=[0, 1], index_col=0)

# This notebook starts where clustering stops, so take the grouping as given.
partition = tsam.aggregate(
    tiny,
    n_clusters=2,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
)
assignments = [int(c) for c in partition.cluster_assignments]

FOCUS = 0
members = [day for day, cluster in enumerate(assignments) if cluster == FOCUS]

print("k=2 Ward assignments, one per day:", assignments)
print(f"focus cluster {FOCUS} = days {members}")

## 1  What comes in: one cluster of four days

The clustering notebooks run the tiny set at **k=3**, which splits the six days into three
*two-member* clusters. Two members are too few to tell the rules apart: with only two periods
the medoid and the maxoid are a tie, and the mean sits exactly halfway between them.

So this notebook uses **k=2**, where Ward separates the two sunny days from everything else:

* cluster 1 = {day0, day1} — the sunny pair
* **cluster 0 = {day2, day3, day4, day5}** — four members, and the one we follow

Four members are enough for every rule to give a different answer, and the cluster contains
**day5**, which carries the series' peak load. Below are its members as the rules see them —
normalized to $[0, 1]$ by [preprocessing](01_preprocessing.ipynb), one row per day, eight
numbers each (2 attributes × 4 timesteps).

In [ ]:
M = D.loc[members].values  # (4 members, 8 = 2 attributes x 4 timesteps)

print("The member matrix M — normalized, one row per day:")
D.loc[members].round(4)

In [ ]:
def period_frame(series, days):
    """One row per day, columns (attribute, timestep), in physical units."""
    rows = {}
    for day in days:
        block = series.iloc[day * N_TIMESTEPS : (day + 1) * N_TIMESTEPS]
        rows[f"day{day}"] = {
            (attr, t): block[attr].iloc[t] for attr in ATTRS for t in range(N_TIMESTEPS)
        }
    frame = pd.DataFrame(rows).T
    frame.columns = pd.MultiIndex.from_tuples(frame.columns, names=["", "TimeStep"])
    return frame


print("The same four days in physical units — solar in W/m², load in MW:")
period_frame(tiny, members)

### The cluster in attribute space

A period is not a point — it is a **path**. With two attributes we can draw it directly: solar
on one axis, load on the other, and one marker per timestep. Each day then traces a path from
`t0` to `t3`, and the labels keep the temporal order visible.

Read the four days as a family: all of them start and end at `solar = 0` (night), swing right
as the sun comes up, and sit at different heights on the load axis. **day5** is the outlier —
it barely leaves the load axis and climbs to 10 MW.

In [ ]:
MEMBER_COLORS = {2: "#8ecae6", 3: "#219ebc", 4: "#ffb703", 5: "#fb8500"}


def attribute_space_figure(title):
    fig = go.Figure()
    fig.update_layout(
        title=title,
        xaxis_title=f"solar [{UNITS['solar']}]",
        yaxis_title=f"load [{UNITS['load']}]",
        legend_title="period",
    )
    return fig


def add_path(fig, solar, load, name, color, width=2, dash=None, label_steps=True):
    """Draw one period as a t0→t3 path in attribute space."""
    fig.add_trace(
        go.Scatter(
            x=solar,
            y=load,
            name=name,
            mode="lines+markers+text" if label_steps else "lines+markers",
            text=[f"t{t}" for t in range(N_TIMESTEPS)] if label_steps else None,
            textposition="top center",
            line={"color": color, "width": width, "dash": dash},
            marker={"size": 8, "color": color},
        )
    )


fig = attribute_space_figure("The four members of cluster 0, each a path from t0 to t3")
for day in members:
    block = tiny.iloc[day * N_TIMESTEPS : (day + 1) * N_TIMESTEPS]
    add_path(fig, block["solar"], block["load"], f"day{day}", MEMBER_COLORS[day])
fig.show()

## 2  What has to come out

One profile for the cluster: **one value per attribute per timestep** — the same eight numbers
any member has. Whatever the rule, the output shape is fixed; only the way those eight numbers
are chosen changes.

That leaves one real question, and it is the one the six rules disagree about: **should the
profile be a period that actually happened, or a constructed one?**

## 3  Inside: the six rules

| tsam name | The profile is… | Chosen |
|---|---|---|
| `mean` | the members' average at each timestep | per timestep |
| `medoid` | the member closest to its cluster-mates | **per period** |
| `maxoid` | the member farthest from the rest of the data | **per period** |
| `distribution` | values re-sorted to keep the cluster's duration curve | per timestep |
| `distribution_minmax` | the same, but each attribute's min and max kept exact | per timestep |
| `minmax_mean` | per attribute: the members' min, max or mean at each timestep | per timestep |

The defaults are set by the clustering method — `kmeans` and `averaging` default to `mean`,
`kmedoids`/`hierarchical`/`contiguous` to `medoid`, and `kmaxoids` to `maxoid` — but any method
can be paired with any of the six.

### `medoid` and `maxoid`: pick a real day

Both select an existing member, so both measure distances — but **they do not measure the same
distances**. The medoid looks **inward**: the member with the smallest total distance to its
own cluster-mates. The maxoid looks **outward**: the member with the largest total distance to
*every period in the dataset*, not just its cluster-mates — it is chosen for being extreme
within the whole series, not merely within its group.

In [ ]:
def distances(X, Y):
    """Euclidean distance between every row of X and every row of Y."""
    return np.sqrt(((X[:, None, :] - Y[None, :, :]) ** 2).sum(-1))


labels = [f"day{d}" for d in members]

within = distances(M, M)
print("Distances among the cluster members (normalized space):")
print(pd.DataFrame(within, index=labels, columns=labels).round(3).to_string())

print("\nmedoid — total distance to the other members (smallest wins):")
for i, day in enumerate(members):
    print(f"  day{day}: {within[i].sum():.3f}")
print(f"  -> medoid = day{members[int(np.argmin(within.sum(axis=0)))]}")

print("\nmaxoid — total distance from all six days in the series (largest wins):")
outward = distances(D.values, M)
for i, day in enumerate(members):
    print(f"  day{day}: {outward[:, i].sum():.3f}")
print(f"  -> maxoid = day{members[int(np.argmax(outward.sum(axis=0)))]}")

The medoid is **day4** and the maxoid is **day5** — different days, as the plot suggested.

### `distribution`: keep the value distribution, not the shape

The distribution rule gives up on matching any member's timing and matches the cluster's
**duration curve** instead — how often each value level occurs. It does this in three moves,
per attribute:

1. **Pool** every value in the cluster — all members, all timesteps (here 4 × 4 = 16 per attribute).
2. **Sort** them and average them down into `n_timesteps` levels: the cluster's duration curve
   at four points.
3. **Order** those levels along the period by ranking the cluster's *mean* profile — the values
   come from the distribution, the ordering is borrowed from the average shape.

With `preserve_minmax` the lowest and highest pooled values are written into the first and last
levels verbatim, so the cluster's true extremes survive step 2's averaging.

### `minmax_mean`: per attribute, per timestep

Each attribute is handled separately: take the members' `min`, `max` or `mean` at each timestep.
Below, `load` is set to `max` and `solar` to `mean` — an envelope on demand, an average on
supply.

In [ ]:
def rule_mean(members_matrix):
    """Per-timestep average across the members."""
    return members_matrix.mean(axis=0)


def rule_medoid(members_matrix):
    """The member with the smallest total distance to its cluster-mates."""
    total = distances(members_matrix, members_matrix).sum(axis=0)
    return members_matrix[int(np.argmin(total))]


def rule_maxoid(members_matrix, all_periods):
    """The member with the largest total distance to every period in the series."""
    total = distances(all_periods, members_matrix).sum(axis=0)
    return members_matrix[int(np.argmax(total))]


def rule_distribution(members_matrix, preserve_minmax=False):
    """Values from the cluster's pooled duration curve, ordered by the mean profile."""
    n_members = members_matrix.shape[0]
    n_attrs = len(ATTRS)
    per_attr = members_matrix.reshape(n_members, n_attrs, N_TIMESTEPS).transpose(
        1, 0, 2
    )

    # 1. pool and 2. sort into n_timesteps duration-curve levels
    pooled = per_attr.reshape(n_attrs, -1).copy()
    pooled.sort(axis=1, kind="stable")
    levels = pooled.reshape(n_attrs, N_TIMESTEPS, n_members).mean(axis=2)
    if preserve_minmax:
        levels[:, 0] = pooled[:, 0]
        levels[:, -1] = pooled[:, -1]

    # 3. order the levels by the ranking of the cluster's mean profile
    order = np.round(per_attr.mean(axis=1), 10).argsort(axis=1, kind="stable")
    profile = np.empty_like(levels)
    profile[np.arange(n_attrs)[:, None], order] = levels
    return profile.ravel()


def rule_minmax_mean(members_matrix, spec):
    """Per attribute, per timestep: the members' min, max or mean."""
    profile = np.zeros(len(ATTRS) * N_TIMESTEPS)
    for a, attr in enumerate(ATTRS):
        start, end = a * N_TIMESTEPS, (a + 1) * N_TIMESTEPS
        block = members_matrix[:, start:end]
        profile[start:end] = {
            "min": block.min(axis=0),
            "max": block.max(axis=0),
            "mean": block.mean(axis=0),
        }[spec[attr]]
    return profile


representatives = {
    "mean": rule_mean(M),
    "medoid": rule_medoid(M),
    "maxoid": rule_maxoid(M, D.values),
    "distribution": rule_distribution(M),
    "distribution_minmax": rule_distribution(M, preserve_minmax=True),
    "minmax_mean": rule_minmax_mean(M, {"solar": "mean", "load": "max"}),
}

print("Six representatives of the same cluster, normalized:")
pd.DataFrame(representatives, index=D.columns).T.round(4)

## 4  What comes out

Back in physical units, the six rules disagree in ways worth reading off carefully — look at
the **load** column, whose true peak in this cluster is **10 MW** (day5, t3).

In [ ]:
def denormalize(profile):
    """Invert the min-max scaling from 01_preprocessing, attribute by attribute."""
    out = profile.reshape(len(ATTRS), N_TIMESTEPS).copy()
    for a, attr in enumerate(ATTRS):
        low, high = tiny[attr].min(), tiny[attr].max()
        out[a] = out[a] * (high - low) + low
    return out.ravel()


physical = pd.DataFrame(
    {name: denormalize(profile) for name, profile in representatives.items()},
    index=D.columns,
).T

print("The six representatives in physical units (solar W/m², load MW):")
print("The cluster's true load peak is 10 MW — watch which rules keep it.")
physical.round(3)

In [ ]:
# Is the representative one of the four days that actually happened?
print("Real period or constructed profile?")
for name, profile in representatives.items():
    match = next(
        (day for day in members if np.allclose(D.loc[day].values, profile, atol=1e-9)),
        None,
    )
    verdict = f"real period  — day{match}" if match is not None else "constructed"
    peak = physical.loc[name, "load"].max()
    print(f"  {name:20s} {verdict:22s} load peak {peak:5.2f} MW")

That table is the whole trade-off in one place:

* **`medoid` and `maxoid` return a period that really happened.** They are chosen *per period* —
  one whole member is lifted out of the cluster, so every internal correlation the day had
  (solar and load at the same hour) survives intact. The price is that a single day has to
  stand in for four.
* **`mean` flattens the peak to 7 MW.** It is chosen *per timestep*: each of the eight numbers
  is averaged independently, so no member's day is preserved — and averaging is exactly what
  destroys extremes. This is what [extreme periods](04_extreme_periods.ipynb) exists to fix.
* **`distribution` reaches 8 MW, `distribution_minmax` recovers the full 10 MW.** Both are
  constructed, but the min/max variant forces the cluster's true extremes into the profile.
* **`minmax_mean` also keeps 10 MW**, because `load` was set to `max` — bought at the cost of
  every load timestep being an envelope rather than a plausible day.

### Per period or per timestep — and why it matters

This is the deepest split between the six rules, and the attribute-space plot below shows it
directly. `medoid` and `maxoid` **land exactly on a member's path**, because they *are* that
member. The other four trace paths **no day ever took**: they are assembled timestep by
timestep. Neither is wrong — but only the first kind can be handed to a downstream model as "a
day that could occur".

In [ ]:
REP_COLORS = {
    "mean": "#000000",
    "medoid": "#2a9d8f",
    "maxoid": "#e63946",
    "distribution": "#6a4c93",
    "distribution_minmax": "#b5179e",
    "minmax_mean": "#457b9d",
}

fig = attribute_space_figure(
    "Members (grey) and the six representatives — click the legend to isolate one"
)
for day in members:
    block = tiny.iloc[day * N_TIMESTEPS : (day + 1) * N_TIMESTEPS]
    add_path(
        fig, block["solar"], block["load"], f"day{day}", "#c9c9c9", label_steps=False
    )

for name, color in REP_COLORS.items():
    row = physical.loc[name]
    add_path(
        fig,
        row["solar"].values,
        row["load"].values,
        name,
        color,
        width=3,
        dash="dot",
        label_steps=False,
    )
fig.show()

## 5  The same thing through tsam

Every profile above was computed by hand from the member matrix. Asking tsam for the same six
representations returns the same numbers.

One detail: `preserve_column_means=False` here. By default tsam **rescales** the representatives
afterwards so the cluster-weighted totals match the original — a correction that would shift
these numbers away from the raw rules. That step is the subject of
[Rescaling](05_rescaling.ipynb), and the `mean` row above is exactly why it is needed.

In [ ]:
# The representation= lever takes the string names above; the typed objects
# (MinMaxMean, Distribution) exist for the options a bare string cannot carry —
# here, which column gets max and which gets mean.
api_names = {
    "mean": "mean",
    "medoid": "medoid",
    "maxoid": "maxoid",
    "distribution": "distribution",
    "distribution_minmax": "distribution_minmax",
    "minmax_mean": MinMaxMean(max_columns=["load"], min_columns=[]),
}

print("hand-computed profile == tsam.aggregate output?")
for name, representation in api_names.items():
    result = tsam.aggregate(
        tiny,
        n_clusters=2,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=representation),
        preserve_column_means=False,
    )
    cluster_id = [int(c) for c in result.cluster_assignments][members[0]]
    profile = result.cluster_representatives.loc[cluster_id]
    from_tsam = np.concatenate([profile[attr].values for attr in ATTRS])
    agrees = np.allclose(from_tsam, denormalize(representatives[name]), atol=1e-9)
    print(f"  {name:20s} {agrees}")

---

**Up next:**

* [Extreme periods](04_extreme_periods.ipynb) — the `mean` row lost the 10 MW peak; extremes
  force a chosen period into the cluster set so it cannot be averaged away.
* [Rescaling](05_rescaling.ipynb) — a non-mean representative distorts the cluster's totals;
  rescaling corrects them, and denormalisation returns the profiles to physical units.

**See also:**

* [Notation and equations](../../reference/notation.md) — every symbol and formula on one page
* [Representations how-to](../../how-to/representations.ipynb) — choosing between these six on a
  realistic series